# URetinex-Net contrast enhancement (BTXRD, deep learning)
Enhances the contrast of every X-ray with **URetinex-Net**, a deep Retinex network, **at its full size** (2440 x 2440), and only then makes it 512 x 512 (the same way as notebook 07). Everything else is the same as in the classical notebooks.
The new images are saved as PNG in **`BTXRD_uretinex_512.zip`**. Inside the zip are the same `train`, `validation` and `test` folders, each with `normal` and `tumor`, as in `BTXRD_512.zip`. Every image stays in the same split as before. The original data is only read, never changed.

**URetinex-Net** (Wu et al., 2022, *URetinex-Net: Retinex-based Deep Unfolding Network for Low-light Image Enhancement*, CVPR) follows the Retinex idea: an image = its detail (reflectance) x its lighting (illumination). The network splits the image into these two, cleans them in 3 rounds, makes the lighting stronger by a chosen factor (`ratio`), and multiplies them back together. This notebook uses the authors' own trained models from github.com/AndersonYong/URetinex-Net. They were made for dark everyday photos, not for X-rays, and they are not trained again here. The supervisor's guide lists deep Retinex networks as the deep learning method for contrast.

**The X-ray is used as a 3-channel image, pixels 0-1**, as in the authors' code. The 3 output channels are averaged back to one gray image.

**The strength (`ratio`) is chosen by this notebook.** The authors recommend 3 to 5 for dark photos and warn that a bigger ratio over-exposes. An X-ray is not a dark photo, and bone is already bright, so too much would turn bone and tumor pure white. So the notebook tries the ratios 1 to 5 on 40 tumor X-rays from the **train images only**. It keeps the ratio with the best tumor contrast (CII) among those that turn at most 1% of the X-ray pixels newly white. If no ratio passes, the notebook stops. The chosen ratio is saved in Drive, so a rerun uses the same one.

**Only the box around the X-ray goes through the network** (plus a margin of 32 pixels). Outside the box the image stays as it is.

**Not tested on X-rays yet:** the network code was copied from the authors' code and checked with random numbers on real X-rays, but the trained models were not downloaded before this notebook was given to you. The first time they meet an X-ray is in your Colab run. So look closely at the table and the figure below before trusting the results.

Time: about 2 hours on a T4 GPU (estimated, not measured on a T4).
## Before you run
1. **In Google Drive**, the folder `BTXRD` (in *My Drive*) must hold `BTXRD_full.zip` and `BTXRD_512.zip`. You already uploaded them for the noise notebooks.
2. **In Colab**: *Runtime > Change runtime type > T4 GPU*, then *Run all*, and allow access to Google Drive.
3. **If Colab disconnects**, just *Run all* again: every finished image is already saved in Drive (folder `BTXRD/BTXRD_uretinex_512`) and is skipped.
4. **At the end**, download `BTXRD_uretinex_512.zip` and `BTXRD_uretinex_512_full_size_scores.csv` (last cell, or from the Drive folder `BTXRD`) and put them in this folder, next to this notebook. Then save this notebook with its outputs: *File > Download > Download .ipynb*. (In Safari, first untick *Settings > General > Open "safe" files after downloading*, or the zip gets unpacked.)

## Settings

In [ ]:
import io
import json
import os
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"           # needed for repeatable GPU results (set before torch starts)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/BTXRD")               # your Google Drive folder with the 2 zips
FULL_ZIP = DRIVE / "BTXRD_full.zip"                        # the full-size images: train, validation, test, Annotations (only read)
OLD_ZIP_DRIVE = DRIVE / "BTXRD_512.zip"                    # the baseline zip (only read)
LOCAL = Path("/content/btxrd_full")                        # the full-size images are unzipped here (the fast local disk)
OLD_ZIP = Path("/content/BTXRD_512.zip")                   # a local copy of the baseline zip
NAME = "BTXRD_uretinex_512"                                # the name of the new zip and of the folder inside it
OUT_DIR = DRIVE / NAME                                     # the finished 512 x 512 PNGs, kept in Drive, so a disconnect loses nothing
NEW_ZIP = Path("/content") / (NAME + ".zip")               # the new zip, made at the end (and copied to Drive)
SPLITS = ["train", "validation", "test"]
MALIGNANT = {"osteosarcoma", "other mt"}               # all other tumor types are benign

SIZE = (512, 512)    # the same size as the baseline
RATIOS = [1, 2, 3, 4, 5]    # URetinex-Net: the strengths to try on train images
RATIO_IMAGES = 40           # URetinex-Net: how many train tumor X-rays are used to choose the strength
MAX_NEW_WHITE = 1.0         # URetinex-Net: a ratio may turn at most 1% of the X-ray pixels newly white
WHITE = 250                 # pixels at or above this count as white
MARGIN = 32                 # URetinex-Net: 32 pixels around the X-ray box
WEIGHTS_URL = "https://raw.githubusercontent.com/AndersonYong/URetinex-Net/7ee2196e093b6a9ae5c20242e48f9d61021b6364/ckpt/"   # the authors' trained models
WEIGHT_FILES = {"init_low.pth": 250453, "unfolding.pth": 3207861, "L_adjust.pth": 1802447}   # file name and its exact size in bytes
WEIGHTS_DIR = Path("/content/uretinex_weights")
RATIO_FILE = DRIVE / "BTXRD_uretinex_ratio.txt"   # the chosen strength, kept in Drive
BLACK = 15           # pixels at or below this are black background
RING = 10            # CII: width of the ring around the tumor, in pixels (on the 512 x 512 image)
RING_FULL = 48       # CII at full size: the same ring, 48 pixels wide on the 2440 x 2440 image (10 x 2440 / 512)
FULL_SCORES = DRIVE / (NAME + "_full_size_scores.csv")     # the full-size scores, one line per image, kept in Drive
SEED = 42            # makes the random parts the same every time

## GPU, settings for repeatable results, and the data
Checks the GPU, unzips the full-size images to the fast local disk, and finds the data folders.

In [ ]:
assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU, then run again"
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True                  # the same GPU calculations every time
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.allow_tf32 = False                    # full 32-bit precision, also on newer GPUs
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)   # repeatable where possible (only warns otherwise)

for f in [FULL_ZIP, OLD_ZIP_DRIVE]:
    assert f.exists(), f"Upload {f.name} to the Google Drive folder {DRIVE} first"

if not (LOCAL / "unzipped.txt").exists():                  # skipped when this runtime already has them
    shutil.rmtree(LOCAL, ignore_errors=True)
    local_zip = Path("/content/BTXRD_full.zip")
    shutil.copy(FULL_ZIP, local_zip)                       # copy from Drive first: the local disk is much faster
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(LOCAL)
    local_zip.unlink()
    (LOCAL / "unzipped.txt").write_text("done")
if not OLD_ZIP.exists():
    shutil.copy(OLD_ZIP_DRIVE, OLD_ZIP)


def find_data(folder):
    for annotations in sorted(folder.rglob("Annotations")):
        root = annotations.parent
        if "__MACOSX" not in root.parts and all((root / split).is_dir() for split in SPLITS):
            return root
    raise FileNotFoundError("BTXRD_full.zip must hold the folders train, validation, test and Annotations")


DATA = find_data(LOCAL)                                    # the original full-size images without duplicates (only read)
ANNOTATIONS = DATA / "Annotations"                         # tumor outlines (only read)
print("Full-size images:", DATA)

## List the images
Every image in the `train`, `validation` and `test` folders, with its split, label (normal or tumor) and group (normal, benign or malignant).

In [ ]:
def tumor_type(path):
    annotation = ANNOTATIONS / (path.stem + ".json")
    if not annotation.exists():
        return ""                                              # normal images have no annotation
    return json.loads(annotation.read_text())["shapes"][0]["label"]


records = []
for split in SPLITS:
    for path in sorted((DATA / split).glob("*/*.jp*g")):       # */ = normal or tumor, files end in .jpeg or .jpg
        if path.name.startswith("._"):
            continue                                           # hidden Mac files, not X-rays
        kind = tumor_type(path)
        if kind == "":
            group = "normal"
        elif kind in MALIGNANT:
            group = "malignant"
        else:
            group = "benign"
        records.append({"image": path.stem, "split": split, "label": path.parent.name, "group": group, "path": path})

images = pd.DataFrame(records)
print(len(images), "images")
print(images.groupby(["split", "label"]).size())

## Check: the same split as `BTXRD_512.zip`
Every image must be in the same split (train, validation or test) and have the same label as in the baseline zip. If not, the notebook stops here.

In [ ]:
old_zip = zipfile.ZipFile(OLD_ZIP)                         # kept open: the baseline images are read from it later
old_names = {name.removeprefix("BTXRD_512/").removesuffix(".png")   # for example train/normal/IMG001868
             for name in old_zip.namelist() if name.endswith(".png")}
names = set(images["split"] + "/" + images["label"] + "/" + images["image"])

print("Images in BTXRD_512.zip:", len(old_names), "| images here:", len(names))
print("Same split and label for every image:", names == old_names)
assert names == old_names, "the split is not the same as in BTXRD_512.zip"

## Functions
Load an image at full size, make an image 512 x 512 (the same way as notebook 07), read a baseline image, and find the X-ray inside the black square.

In [ ]:
def load(path):
    image = Image.open(path).convert("L")                      # gray, pixels 0 (black) to 255 (white)
    return np.array(image)                                     # full size: 2440 x 2440


def shrink(image):
    image = Image.fromarray(image).resize(SIZE, Image.LANCZOS)  # 512 x 512, the same way as notebook 07
    return np.array(image)


def baseline(item):
    path = f"BTXRD_512/{item['split']}/{item['label']}/{item['image']}.png"
    return np.array(Image.open(io.BytesIO(old_zip.read(path))))   # the baseline image the models were trained on


def xray_box(image):
    rows = np.where((image > BLACK).any(axis=1))[0]            # rows with something brighter than black
    cols = np.where((image > BLACK).any(axis=0))[0]            # the same for columns
    return rows[0], rows[-1] + 1, cols[0], cols[-1] + 1        # top, bottom, left, right

## Check: Colab makes 512 x 512 images exactly like the baseline
The baseline images were made on the Mac. If Colab's image library shrank images even a little differently, part of the difference would come from that and not from the method. So 25 images are shrunk here and must be exactly the same as in `BTXRD_512.zip`.

In [ ]:
sample = images.sample(25, random_state=SEED)
same = [np.array_equal(shrink(load(item["path"])), baseline(item)) for _, item in sample.iterrows()]
print("Made 512 x 512 exactly like BTXRD_512.zip:", sum(same), "of", len(same))
assert all(same), "This Pillow version resizes differently: run  !pip install pillow==12.3.0  then Runtime > Restart, and run again"

## The URetinex-Net model
The same networks as in the authors' code, loaded with their trained weights. `strict=True` checks that the files and the networks match exactly.

In [ ]:
import argparse

torch.serialization.add_safe_globals([argparse.Namespace])     # the authors' files also hold their settings as this kind of object


class Decom(nn.Module):                                        # network/decom.py: the first split into detail and lighting
    def __init__(self):
        super().__init__()
        self.decom = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 32, 3, 1, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 32, 3, 1, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 4, 3, 1, 1), nn.ReLU())

    def forward(self, x):
        output = self.decom(x)
        return output[:, 0:3, :, :], output[:, 3:4, :, :]      # detail (3 channels), lighting (1 channel)


class SELayer(nn.Module):                                      # network/restoration.py
    def __init__(self, channel=64, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(channel, channel // reduction, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(channel // reduction, channel, bias=False), nn.Sigmoid())

    def forward(self, x):
        batch, channels, _, _ = x.size()
        y = self.fc(self.avg_pool(x).view(batch, channels)).view(batch, channels, 1, 1)
        return x * y.expand_as(x)


class HalfDnCNNSE(nn.Module):                                  # network/restoration.py: cleans the detail
    def __init__(self, concat_L):
        super().__init__()
        self.concat_L = concat_L
        if concat_L:
            self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)
            self.conv2 = nn.Conv2d(1, 32, 3, 1, 1)
        else:
            self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.se_layer = SELayer(channel=64)
        self.conv3 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv4 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv5 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv6 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv7 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv8 = nn.Conv2d(64, 3, 3, 1, 1)

    def forward(self, r, l):
        if self.concat_L:
            features = torch.cat([F.relu(self.conv1(r)), F.relu(self.conv2(l))], dim=1)
        else:
            features = F.relu(self.conv1(r))
        x = F.relu(self.conv3(self.se_layer(features)))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))
        x = F.relu(self.conv7(x))
        return r + self.conv8(x)


class IlluminationAlone(nn.Module):                            # Illumination_Alone in network/illumination_enhance.py: cleans the lighting
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 5, 1, 2)
        self.conv2 = nn.Conv2d(32, 32, 5, 1, 2)
        self.conv3 = nn.Conv2d(32, 32, 5, 1, 2)
        self.conv4 = nn.Conv2d(32, 32, 5, 1, 2)
        self.conv5 = nn.Conv2d(32, 1, 1, 1, 0)

    def forward(self, l):
        x = F.leaky_relu(self.conv1(l), 0.2)
        x = F.leaky_relu(self.conv2(x), 0.2)
        x = F.leaky_relu(self.conv3(x), 0.2)
        x = F.leaky_relu(self.conv4(x), 0.2)
        return F.relu(self.conv5(x))


class AdjustNaive(nn.Module):                                  # Adjust_naive in network/illumination_adjustment.py: makes the lighting stronger
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 32, 5, 1, 2)
        self.conv2 = nn.Conv2d(32, 32, 5, 1, 2)
        self.conv3 = nn.Conv2d(32, 32, 5, 1, 2)
        self.conv4 = nn.Conv2d(32, 1, 5, 1, 2)

    def forward(self, l, alpha):
        x = self.conv1(torch.cat([l, alpha], dim=1))
        x = self.conv2(F.leaky_relu(x, 0.2))
        x = self.conv3(F.leaky_relu(x, 0.2))
        x = self.conv4(F.leaky_relu(x, 0.2))
        return F.relu(x)


def trained_file(name):
    path = WEIGHTS_DIR / name
    if not path.exists():
        WEIGHTS_DIR.mkdir(exist_ok=True)
        torch.hub.download_url_to_file(WEIGHTS_URL + name, str(path))   # stops with an error if the download fails
    assert path.stat().st_size == WEIGHT_FILES[name], f"{name} is not the expected file"
    return torch.load(path, map_location="cpu", weights_only=True)    # safe loading: only numbers and the authors' settings object


decom_file, unfolding_file, adjust_file = [trained_file(name) for name in WEIGHT_FILES]
settings = unfolding_file["opts"]                              # the authors' saved settings: number of rounds and the weights of each round
assert settings.R_model == "HalfDnCNNSE" and settings.L_model == "Illumination_Alone" and adjust_file["opts"].A_model == "naive"

decom_net = Decom()
detail_net = HalfDnCNNSE(settings.concat_L)
lighting_net = IlluminationAlone()
adjust_net = AdjustNaive()
decom_net.load_state_dict(decom_file["state_dict"]["model_R"], strict=True)   # strict: every layer must match exactly (the authors store it under this name)
detail_net.load_state_dict(unfolding_file["state_dict"]["model_R"], strict=True)
lighting_net.load_state_dict(unfolding_file["state_dict"]["model_L"], strict=True)
adjust_net.load_state_dict(adjust_file["state_dict"]["model_A"], strict=True)
for network in [decom_net, detail_net, lighting_net, adjust_net]:
    network.to(DEVICE).eval()
print("URetinex-Net loaded, rounds:", settings.round, "| the detail network uses the lighting too:", settings.concat_L)


def split_detail_lighting(x):                                  # unfolding in the authors' test.py
    for t in range(settings.round):
        if t == 0:
            p, q = decom_net(x)
        else:
            weight_p = settings.gamma + settings.Roffset * t
            weight_q = settings.lamda + settings.Loffset * t
            p = (x * q + weight_p * detail) / (weight_p + q * q)   # network/Math_Module.py, class P
            q = ((x * p).sum(dim=1, keepdim=True) + weight_q * lighting) / ((p * p).sum(dim=1, keepdim=True) + weight_q)   # class Q
        detail = detail_net(p, q)
        lighting = lighting_net(q)
    return detail, lighting


def stronger_lighting(detail, lighting, ratio):
    enhanced = detail * adjust_net(lighting, torch.ones_like(lighting) * ratio)   # image = detail x stronger lighting
    return (enhanced.mean(dim=1)[0].clamp(0, 1) * 255).round().to(torch.uint8).cpu().numpy()   # 3 channels to 1 gray image, back to 0-255


def box_with_margin(image):
    top, bottom, left, right = xray_box(image)                 # the box around the X-ray ...
    return max(top - MARGIN, 0), min(bottom + MARGIN, image.shape[0]), max(left - MARGIN, 0), min(right + MARGIN, image.shape[1])   # ... with a margin


def to_network(crop):
    return torch.from_numpy(crop).float().div(255).to(DEVICE)[None, None].repeat(1, 3, 1, 1)   # 1 x 3 x H x W, pixels 0-1

## Measuring functions
The same functions as in the classical notebooks.
- PSNR and SSIM: how much the image changed (higher = less change). Measured on the X-ray only, not the black square.
- CII: how much more the tumor stands out from the ring around it. Above 1 = it stands out more.

In [ ]:
def tumor_mask(name):
    annotation = json.loads((ANNOTATIONS / (name + ".json")).read_text())
    scale = SIZE[0] / annotation["imageWidth"]                 # the outline was drawn on the 2440 x 2440 image
    mask = np.zeros((SIZE[1], SIZE[0]), dtype=np.uint8)
    for shape in annotation["shapes"]:
        if shape["shape_type"] == "polygon":                   # the outline (the rectangles are only boxes)
            points = np.round(np.array(shape["points"]) * scale).astype(np.int32)
            cv2.fillPoly(mask, [points], 1)
    return mask == 1                                           # True inside the tumor


def ring_around(tumor, before):
    kernel = np.ones((2 * RING + 1, 2 * RING + 1), dtype=np.uint8)
    bigger = cv2.dilate(tumor.astype(np.uint8), kernel) == 1   # the tumor made RING pixels bigger
    return bigger & ~tumor & (before > BLACK)                  # the ring: not the tumor, not the black background


def contrast(image, tumor, ring):
    t = image[tumor].mean()                                    # tumor brightness
    r = image[ring].mean()                                     # ring brightness
    return abs(t - r) / (t + r)                                # abs: a tumor can be darker or brighter than the bone


def cii(before, after, tumor, ring):
    if tumor.sum() == 0 or ring.sum() == 0:
        return np.nan                                          # cannot be measured
    contrast_before = contrast(before, tumor, ring)
    if contrast_before == 0:
        return np.nan                                          # cannot divide by 0
    return contrast(after, tumor, ring) / contrast_before


def psnr_ssim(before, after):
    top, bottom, left, right = xray_box(before)                # the X-ray only
    a, b = before[top:bottom, left:right], after[top:bottom, left:right]
    return peak_signal_noise_ratio(a, b, data_range=255), structural_similarity(a, b, data_range=255)

## Measuring functions at full size
The same tumor outline and ring, but on the 2440 x 2440 image: the outline is not made smaller, and the ring is 48 pixels wide (the same area as the 10-pixel ring at 512 x 512), as in the classical `*_full_size_scores` notebooks.

In [ ]:
def tumor_mask_full(name, size):
    annotation = json.loads((ANNOTATIONS / (name + ".json")).read_text())
    scale = size[1] / annotation["imageWidth"]                 # 1.0: the outline was drawn on the 2440 x 2440 image
    mask = np.zeros(size, dtype=np.uint8)
    for shape in annotation["shapes"]:
        if shape["shape_type"] == "polygon":                   # the outline (the rectangles are only boxes)
            points = np.round(np.array(shape["points"]) * scale).astype(np.int32)
            cv2.fillPoly(mask, [points], 1)
    return mask == 1                                           # True inside the tumor


def ring_around_full(tumor, before):
    kernel = np.ones((2 * RING_FULL + 1, 2 * RING_FULL + 1), dtype=np.uint8)
    bigger = cv2.dilate(tumor.astype(np.uint8), kernel) == 1   # the tumor made RING_FULL pixels bigger
    return bigger & ~tumor & (before > BLACK)                  # the ring: not the tumor, not the black background

## Choose the strength (`ratio`) on train images
Every ratio from 1 to 5 is tried on 40 tumor X-rays from the train images. The table shows the middle CII (above 1 = the tumor stands out more), how many X-ray pixels newly became white, and the X-ray brightness. The ratio with the best CII among those with at most 1% new white pixels is kept. **Look at this table and at the figure below.** If the images look wrong, stop and send them to Claude.

In [ ]:
def try_ratios():
    picked = images[(images["split"] == "train") & (images["label"] == "tumor")].sample(RATIO_IMAGES, random_state=SEED)
    tried = []
    for _, item in picked.iterrows():
        raw = load(item["path"])
        top, bottom, left, right = box_with_margin(raw)
        with torch.inference_mode():
            detail, lighting = split_detail_lighting(to_network(raw[top:bottom, left:right]))
            for ratio in RATIOS:
                full = raw.copy()
                full[top:bottom, left:right] = stronger_lighting(detail, lighting, ratio)
                xray = raw > BLACK
                tumor = tumor_mask_full(item["image"], raw.shape)
                tried.append({"ratio": ratio, "CII": cii(raw, full, tumor, ring_around_full(tumor, raw)),
                              "new white pixels (%)": 100 * ((full[xray] >= WHITE).mean() - (raw[xray] >= WHITE).mean()),
                              "X-ray brightness": full[xray].mean(), "original brightness": raw[xray].mean()})
    return pd.DataFrame(tried).groupby("ratio").agg({"CII": "median", "new white pixels (%)": "mean", "X-ray brightness": "mean", "original brightness": "mean"}).round(3)


ratio_table = try_ratios()
print("Train tumor X-rays only:")
print(ratio_table)
assert ratio_table.notna().all().all(), "The network gave invalid numbers: send this output to Claude before going on"

allowed = ratio_table[ratio_table["new white pixels (%)"] <= MAX_NEW_WHITE]
assert len(allowed) > 0, "Every ratio turns too much of the X-ray white: send this output to Claude before going on"
best_ratio = int(allowed["CII"].idxmax())

if RATIO_FILE.exists():
    RATIO = int(RATIO_FILE.read_text())                        # chosen in an earlier run: stay with it, so all images are made the same way
    print("Using the ratio chosen earlier:", RATIO, "| chosen now:", best_ratio)
else:
    RATIO = best_ratio
    RATIO_FILE.write_text(str(RATIO))
    print("Chosen ratio:", RATIO)


def uretinex(image):
    top, bottom, left, right = box_with_margin(image)
    with torch.inference_mode():
        detail, lighting = split_detail_lighting(to_network(image[top:bottom, left:right]))
        result = image.copy()                                  # outside the box the image stays as it is
        result[top:bottom, left:right] = stronger_lighting(detail, lighting, RATIO)
    return result

## Before and after: 2 normal, 2 benign, 2 malignant
Both at 512 x 512, the size the models see. Before = the baseline image (the same as in `BTXRD_512.zip`).

In [ ]:
picks = pd.concat([images[images["group"] == group].sample(2, random_state=42)
                   for group in ["normal", "benign", "malignant"]])
titles = ["before (baseline)", "after URetinex-Net"]

fig, axes = plt.subplots(len(picks), 2, figsize=(8, 4.4 * len(picks)))
for row, (_, pick) in zip(axes, picks.iterrows()):
    before = baseline(pick)                                    # the baseline image
    after = shrink(uretinex(load(pick['path'])))               # URetinex-Net on the full image, then 512 x 512
    top, bottom, left, right = xray_box(before)                # cut away the black square, so the bone is big enough to see
    for ax, picture, title in zip(row, [before, after], titles):
        ax.imshow(picture[top:bottom, left:right], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{pick['image']} ({pick['group']})\n{title}")
        ax.axis("off")

plt.tight_layout()
plt.show()

## Make every image
For every image: URetinex-Net on the full image, then 512 x 512, then saved as PNG in the Drive folder `BTXRD/BTXRD_uretinex_512`.
While the full-size result is there, its **full-size scores** are measured too (the full image after URetinex-Net against the original full image) and saved as one line per image in `BTXRD/BTXRD_uretinex_512_full_size_scores.csv`. So the method runs only once.
Images that are already done are skipped, so after a disconnect this continues where it stopped.

In [ ]:
def png_path(item):
    return OUT_DIR / item["split"] / item["label"] / (item["image"] + ".png")


def scored_images():
    if not FULL_SCORES.exists():
        return set()
    return set(pd.read_csv(FULL_SCORES)["image"])             # images that already have their full-size scores


scored = scored_images()
todo = [item for _, item in images.iterrows() if not png_path(item).exists() or item["image"] not in scored]
print(len(images) - len(todo), "images already done,", len(todo), "to do")

start = time.time()
for n, item in enumerate(todo, start=1):
    raw = load(item["path"])                                   # the original X-ray at full size
    full = uretinex(raw)                                       # URetinex-Net on the full image (2440 x 2440)
    after = shrink(full)                                       # then 512 x 512

    psnr, ssim = psnr_ssim(raw, full)                          # full-size scores: the full image after the method vs the original
    row = {"image": item["image"], "label": item["label"], "PSNR": psnr, "SSIM": ssim, "CII": np.nan}
    if item["label"] == "tumor":
        tumor = tumor_mask_full(item["image"], raw.shape)
        row["CII"] = cii(raw, full, tumor, ring_around_full(tumor, raw))
    pd.DataFrame([row]).to_csv(FULL_SCORES, mode="a", header=not FULL_SCORES.exists(), index=False)   # one line per image, in Drive

    target = png_path(item)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(".tmp")
    Image.fromarray(after).save(temporary, format="PNG")
    os.replace(temporary, target)                              # renamed only when complete, so a half-written image is never kept

    if n % 100 == 0 or n == len(todo):
        minutes_left = (time.time() - start) / n * (len(todo) - n) / 60
        print(f"{n} of {len(todo)} done, about {minutes_left:.0f} minutes left")

done = sum(1 for _, item in images.iterrows() if png_path(item).exists())
scored = scored_images()
print(done, "of", len(images), "images are in", OUT_DIR, "|", len(scored & set(images["image"])), "have full-size scores")
assert done == len(images) and set(images["image"]) <= scored, "not all images are done: run this cell again"

## Make the zip
Puts all images in `BTXRD_uretinex_512.zip` (`BTXRD_uretinex_512/split/label/name.png`), the same layout as `BTXRD_512.zip`, and also copies it to Drive.

In [ ]:
with zipfile.ZipFile(NEW_ZIP, "w") as new_zip:                 # "w": makes a new zip (an old one with the same name is replaced)
    for _, item in images.iterrows():
        source = OUT_DIR / item["split"] / item["label"] / (item["image"] + ".png")
        new_zip.write(source, f"{NAME}/{item['split']}/{item['label']}/{item['image']}.png")

shutil.copy(NEW_ZIP, DRIVE / NEW_ZIP.name)                     # also keep the zip in Google Drive
print(len(images), "images saved in", NEW_ZIP, "and in", DRIVE / NEW_ZIP.name)

## Check the new zip
Counts the images in each folder, checks that all are 512 x 512 gray images, and that every image is in the same split and label as in `BTXRD_512.zip`.

In [ ]:
with zipfile.ZipFile(NEW_ZIP) as new_zip:
    files = [name for name in new_zip.namelist() if name.endswith(".png")]
    sizes, modes = Counter(), Counter()
    for name in files:
        image = Image.open(new_zip.open(name))
        sizes[image.size] += 1
        modes[image.mode] += 1                                 # L = gray

for folder, count in sorted(Counter(name.rsplit("/", 1)[0] for name in files).items()):
    print(folder, count)
print("Sizes:", dict(sizes), "| modes:", dict(modes))
new_names = {name.removeprefix(NAME + "/").removesuffix(".png") for name in files}
print("Same images, split and label as BTXRD_512.zip:", new_names == old_names)
print("Zip size:", round(NEW_ZIP.stat().st_size / 1e6), "MB")

## Measure every image at 512 x 512
PSNR, SSIM and CII of every image in the new zip against its baseline image, as in the classical notebooks. CII is only for tumor images.

In [ ]:
rows = []
with zipfile.ZipFile(NEW_ZIP) as new_zip:
    for _, item in images.iterrows():
        before = baseline(item)                                # the baseline image
        after = np.array(Image.open(io.BytesIO(new_zip.read(f"{NAME}/{item['split']}/{item['label']}/{item['image']}.png"))))

        psnr, ssim = psnr_ssim(before, after)
        row = {"image": item["image"], "label": item["label"], "PSNR": psnr, "SSIM": ssim, "CII": np.nan}
        if item["label"] == "tumor":
            tumor = tumor_mask(item["image"])
            row["CII"] = cii(before, after, tumor, ring_around(tumor, before))
        rows.append(row)

scores = pd.DataFrame(rows)
print(len(scores), "images measured")

## Results at 512 x 512
The middle value (median) of each score: the image after URetinex-Net compared with the baseline image. CII only counts the tumor images.

In [ ]:
scores[["PSNR", "SSIM", "CII"]].median().round(3)

## Results at full size
The middle value (median) of each score at full size: the full image after URetinex-Net compared with the original full image. CII only counts the tumor images.

In [ ]:
full_scores = pd.read_csv(FULL_SCORES).drop_duplicates("image", keep="last")   # one line per image (a disconnect can repeat one)
full_scores = full_scores[full_scores["image"].isin(images["image"])]
print(len(full_scores), "images with full-size scores")
assert len(full_scores) == len(images)
full_scores[["PSNR", "SSIM", "CII"]].median().round(3)

## Download the zip and the full-size scores

In [ ]:
from google.colab import files                                 # imported again: the zip check above uses the name "files" for its list

files.download(str(NEW_ZIP))                                   # or download both from the Google Drive folder BTXRD
files.download(str(FULL_SCORES))